# Notebook 06 — Scoring Matrices: PAM and BLOSUM

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 06 of 18  
**Time estimate:** 75 minutes

> "What is a BLOSUM matrix?" is asked in nearly every bioinformatics interview.
> Know the biological rationale, not just the definition.

---
## Step 1 — Motivation

In NB04–NB05 we used +1/-1 for match/mismatch. But not all mismatches are equal:
leucine→isoleucine (both hydrophobic, similar size) is far more tolerated in proteins
than lysine→aspartate (opposite charges). A scoring matrix encodes this biology.
BLAST uses BLOSUM62 by default — knowing what that means is a Track A baseline.

---
## Step 2 — Intuition

**PAM (Point Accepted Mutation):** Built from very closely related protein pairs.
Asks: "in one step of accepted evolution, how often does amino acid A change to B?"
PAM1 = 1% divergence. PAM250 = matrix extrapolated to ~80% divergence.

**BLOSUM (BLOcks SUbstitution Matrix):** Built from conserved blocks in distantly
related proteins. The number (BLOSUM62, BLOSUM80) is the % identity threshold:
BLOSUM62 = derived from blocks with ≤62% identity. Lower number → more divergent
sequences → better for detecting distant homologs.

**Log-odds score:** $s(a,b) = \log_2 \frac{P(\text{observe pair } a,b \mid \text{homologous})}{P(\text{observe pair } a,b \mid \text{random})}$

Positive = more likely than chance (conserved), negative = less likely (selected against).

---
## Step 3 — Biological Background

Amino acid substitutions are constrained by structure and function:
- Conservative substitutions: similar physicochemical properties (charge, size,
  hydrophobicity) — frequently observed in alignments
- Non-conservative substitutions: opposite properties — rarely observed, often
  deleterious

BLOSUM62 reflects this: Leu↔Ile scores +2 (common, conservative), Asp↔Lys scores
-1 (opposite charge, uncommon). The matrix entries are therefore a direct readout
of evolutionary constraint.

---
## Step 4 — Mathematical Explanation

**BLOSUM construction:**

1. Start with the BLOCKS database: multiply-aligned conserved protein regions
2. Cluster sequences within each block at the identity threshold (e.g., 62% for BLOSUM62)
3. Count all pairs of aligned amino acids $(a, b)$ across all blocks
4. Compute:
   - $q_a$ = frequency of amino acid $a$
   - $p_{ab}$ = observed frequency of aligned pair $(a, b)$
   - $e_{ab}$ = expected frequency = $q_a \cdot q_b$ (for $a \neq b$); $q_a^2$ (for $a = b$)
5. Log-odds score: $s(a,b) = \lambda \cdot \log_2 \frac{p_{ab}}{e_{ab}}$ (rounded to nearest integer)

where $\lambda$ is a scaling factor to give integer scores.

**Choosing the right matrix:**
- BLOSUM80: closely related sequences (>80% identity)
- BLOSUM62: default — sequences ~40-60% similar; the most commonly used
- BLOSUM45: distant sequences (<40% identity)

In [ ]:
# Step 6 — Python Implementation
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from Bio.Align import substitution_matrices

# Load BLOSUM62 from Biopython
blosum62 = substitution_matrices.load('BLOSUM62')
amino_acids = list(blosum62.alphabet)
# Keep only standard 20 AAs
std_aa = list('ACDEFGHIKLMNPQRSTVWY')

# Build numpy matrix for standard AAs
n_aa = len(std_aa)
M = np.zeros((n_aa, n_aa))
for i, a in enumerate(std_aa):
    for j, b in enumerate(std_aa):
        M[i, j] = blosum62[a][b]

print("BLOSUM62 for selected amino acid pairs:")
pairs = [
    ('L', 'I', 'Leu-Ile (both hydrophobic, similar)'),
    ('V', 'L', 'Val-Leu (hydrophobic, similar size)'),
    ('D', 'K', 'Asp-Lys (opposite charge)'),
    ('W', 'P', 'Trp-Pro (structurally incompatible)'),
    ('S', 'T', 'Ser-Thr (both polar, similar)'),
    ('G', 'A', 'Gly-Ala (smallest AAs)'),
]
for a, b, desc in pairs:
    score = blosum62[a][b]
    print(f"  {a}-{b}: {score:>3}  ({desc})")

print()
print("Diagonal (same AA):")
for aa in ['W', 'C', 'M', 'A', 'G', 'S']:
    print(f"  {aa}-{aa}: {blosum62[aa][aa]:>3}")

In [ ]:
# Implement a simplified log-odds matrix from a toy alignment
from collections import Counter
import math


def compute_log_odds_matrix(
    aligned_pairs: list[tuple[str, str]],
    pseudocount: float = 0.5
) -> dict[tuple[str, str], float]:
    """
    Given a list of (aa1, aa2) aligned pairs,
    compute log2(observed/expected) for each pair.
    """
    # Count individual amino acids
    all_aa = [a for pair in aligned_pairs for a in pair]
    total = len(all_aa)
    freq = Counter(all_aa)
    q = {aa: (freq[aa] + pseudocount) / (total + pseudocount * len(freq)) for aa in freq}

    # Count aligned pairs (treat a,b and b,a as the same)
    pair_counts = Counter()
    for a, b in aligned_pairs:
        key = tuple(sorted([a, b]))
        pair_counts[key] += 1
    total_pairs = sum(pair_counts.values())

    log_odds = {}
    for (a, b), count in pair_counts.items():
        observed = (count + pseudocount) / (total_pairs + pseudocount * len(pair_counts))
        if a == b:
            expected = q.get(a, 0.05) ** 2
        else:
            expected = 2 * q.get(a, 0.05) * q.get(b, 0.05)
        if expected > 0:
            log_odds[(a, b)] = math.log2(observed / expected)
            log_odds[(b, a)] = log_odds[(a, b)]
    return log_odds


# Toy example: simulate aligned pairs from a conserved protein block
rng = np.random.default_rng(42)

# Conservative pairs are over-represented; radical changes are rare
conservative_pairs = [('L','I'), ('L','V'), ('I','V'), ('D','E'), ('K','R'), ('S','T')]
radical_pairs = [('D','K'), ('W','G'), ('P','C')]
same_pairs = [(aa,aa) for aa in 'ACDEFGHIKLMNPQRSTVWY']

# Generate synthetic training pairs
pairs = []
for aa in 'ACDEFGHIKLMNPQRSTVWY':
    pairs.extend([(aa, aa)] * 10)  # same AA most common
for a, b in conservative_pairs:
    pairs.extend([(a, b)] * 5)  # conservative changes common
for a, b in radical_pairs:
    pairs.extend([(a, b)] * 1)  # radical changes rare

log_odds = compute_log_odds_matrix(pairs)

print("Toy log-odds matrix (should mirror BLOSUM pattern):")
for a, b, desc in [
    ('L', 'I', 'conservative'),
    ('D', 'E', 'conservative'),
    ('D', 'K', 'radical'),
    ('W', 'G', 'radical'),
]:
    score = log_odds.get((a, b), float('nan'))
    print(f"  {a}-{b} ({desc}): {score:.2f}")

In [ ]:
# Step 7 — Visualization: BLOSUM62 heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel A: BLOSUM62 heatmap
ax = axes[0]
im = ax.imshow(M, cmap='RdYlGn', vmin=-4, vmax=11, aspect='auto')
ax.set_xticks(range(n_aa))
ax.set_xticklabels(std_aa, fontsize=9)
ax.set_yticks(range(n_aa))
ax.set_yticklabels(std_aa, fontsize=9)
ax.set_title('BLOSUM62 Substitution Matrix\nGreen=positive (favored), Red=negative (disfavored)')
plt.colorbar(im, ax=ax)

# Annotate select cells
for (a, b, label) in [('L','I',''), ('D','K',''), ('W','W','')]:
    if a in std_aa and b in std_aa:
        i_idx = std_aa.index(a)
        j_idx = std_aa.index(b)
        ax.add_patch(plt.Rectangle((j_idx-0.5, i_idx-0.5), 1, 1,
                                    fill=False, edgecolor='black', lw=2))

# Panel B: Distribution of BLOSUM62 scores
ax2 = axes[1]
scores = M.flatten()
ax2.hist(scores, bins=25, color='steelblue', edgecolor='white')
ax2.axvline(0, color='red', ls='--', label='Score = 0')
ax2.set_xlabel('BLOSUM62 score')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of BLOSUM62 scores\nMost are negative (chance-level or worse)')
ax2.legend()

# Add text annotations
diag_mean = np.diag(M).mean()
off_diag = M[~np.eye(n_aa, dtype=bool)]
ax2.axvline(diag_mean, color='green', ls='--', label=f'Diagonal mean={diag_mean:.1f}')
ax2.legend()

plt.tight_layout()
plt.savefig('blosum62_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Diagonal mean (self-substitution): {diag_mean:.2f}")
print(f"Off-diagonal mean (mismatches): {off_diag.mean():.2f}")

In [ ]:
# Use BLOSUM62 in NW alignment

def needleman_wunsch_blosum(
    seq1: str,
    seq2: str,
    matrix,
    gap: int = -4
) -> tuple[int, str, str]:
    n, m = len(seq1), len(seq2)
    F = np.zeros((n + 1, m + 1), dtype=float)
    for i in range(n + 1): F[i, 0] = i * gap
    for j in range(m + 1): F[0, j] = j * gap

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sc = matrix[seq1[i-1]][seq2[j-1]]
            F[i, j] = max(
                F[i-1, j-1] + sc,
                F[i-1, j] + gap,
                F[i, j-1] + gap
            )

    # Traceback
    align1, align2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            sc = matrix[seq1[i-1]][seq2[j-1]]
            if abs(F[i, j] - (F[i-1, j-1] + sc)) < 1e-9:
                align1.append(seq1[i-1]); align2.append(seq2[j-1]); i -= 1; j -= 1; continue
        if i > 0 and abs(F[i, j] - (F[i-1, j] + gap)) < 1e-9:
            align1.append(seq1[i-1]); align2.append('-'); i -= 1
        else:
            align1.append('-'); align2.append(seq2[j-1]); j -= 1

    return int(F[n, m]), ''.join(reversed(align1)), ''.join(reversed(align2))


# Protein alignment with BLOSUM62
prot1 = 'MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL'
prot2 = 'MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLK'

score, a1, a2 = needleman_wunsch_blosum(prot1[:30], prot2[:30], blosum62)
print(f"BLOSUM62 NW alignment of first 30 AAs:")
print(f"Score: {score}")
print(f"seq1: {a1}")
mid = ''.join('|' if a==b else (' ' if '-' in (a,b) else '.') for a,b in zip(a1,a2))
print(f"      {mid}")
print(f"seq2: {a2}")

---
## Step 8 — Exercises

See `exercises/06_scoring_matrix_exercises.md`.

1. Look up the BLOSUM62 score for K-R (Lys-Arg). Is it positive or negative?
   Explain why in terms of amino acid chemistry.
2. Compare NW alignment scores for the same pair using BLOSUM62 vs. PAM250. Which
   one gives a higher score for distantly related proteins? Why?
3. The BLOSUM62 diagonal entry for Trp (W) is 11. For Gly (G) it is 6. Explain
   why Trp self-substitution is scored so much higher.
4. What is the log-odds interpretation of a BLOSUM62 score of -4?

---
## Step 10 — Quiz

1. What does the 62 in BLOSUM62 mean?
2. A BLOSUM62 score of 0 means what, exactly?
3. Which BLOSUM matrix should you use for closely related sequences? Which for
   distant ones?
4. How is PAM250 derived from PAM1?
5. Why are most BLOSUM62 off-diagonal entries negative?

---
## Step 12 — Reflection

> *[Explain in one paragraph: how is BLOSUM62 constructed, and why does the 62%
> identity threshold matter for the matrix's usefulness?]*

---
## Step 13 — Papers Referenced

- Henikoff & Henikoff (1992) "Amino acid substitution matrices from protein blocks."
  *PNAS* 89(22). The BLOSUM paper. Pass-2 read.
- Dayhoff et al. (1978) "A model of evolutionary change in proteins." The PAM paper.

---
*Next: `07_gap_penalties_linear_vs_affine.ipynb`*